# CNN 第1週 — PyTorchで手書き数字を分類しよう
## MNIST：0〜9の画像分類と 5-Fold Cross Validation

この実習では、手書き数字の画像を入力し、  
画像が **0〜9のどの数字であるか** を予測するCNNを作ります。

使用するフレームワークは **PyTorch** です。

---

## この実習で学ぶこと

1. MNIST画像を読み込む
2. 画像をPyTorchのTensorに変換する
3. `Dataset` と `DataLoader` の役割を理解する
4. PyTorchでCNNを定義する
5. Forward、Loss、Backward、Weight Updateを実行する
6. **5-Fold Cross Validation** を行う
7. FoldごとのAccuracyと平均・標準偏差を確認する
8. 最後に、学習に使っていないTestデータで1回だけ評価する
9. Confusion Matrixと間違えた画像を確認する

> このノートブックでは、授業時間内に実行しやすいように  
> 5-Fold部分では学習データの一部を使う **FAST MODE** を用意しています。


## 0. Colabの実行環境を確認する

CNNの学習はCPUでも可能ですが、GPUを使うと短時間で実行できます。

Google ColabでGPUを使う場合は、次の順番で設定します。

**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → GPU**

このセルでは、PyTorchのバージョンと利用するデバイスを確認します。


In [ ]:
import torch

print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("使用するデバイス:", device)


## 1. ライブラリを読み込む

今回使う主なライブラリは次のとおりです。

- `torch`：Tensor計算とニューラルネットワーク
- `torch.nn`：CNNのLayerとLoss
- `torchvision`：MNISTデータセットと画像変換
- `DataLoader`：画像をBatch単位でモデルに渡す
- `StratifiedKFold`：各クラスの割合を保つ5-Fold分割
- `matplotlib`：画像や結果のグラフ表示


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


## 2. 乱数を固定する

ニューラルネットワークでは、

- Weightの初期値
- データを並べ替える順番
- Train / Validationの分割

などに乱数が使われます。

できるだけ同じ結果を再現しやすくするため、乱数のSeedを固定します。

> GPUの計算方法などによっては、完全に同一の結果にならない場合もあります。


In [ ]:
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("Random seed:", SEED)


# PART 1 — MNISTデータを理解する

## 3. MNISTとは

MNISTは、**0〜9の手書き数字画像**を集めた画像分類データセットです。

- Train画像：60,000枚
- Test画像：10,000枚
- 画像サイズ：28 × 28 pixel
- 色：Grayscale
- クラス数：10
- 正解ラベル：0〜9の整数

今回の入力 **X** は数字の画像、  
正解 **y** はその画像に書かれている数字です。


## 4. 画像をTensorに変換する

MNISTの元画像は画像ファイルとして読み込まれます。

`transforms.ToTensor()` は、画像をPyTorchのTensorに変換します。

さらに、pixel値を次のように変換します。

```text
元のpixel値：0〜255
      ↓
Tensorの値：0.0〜1.0
```

今回のMNIST画像1枚のshapeは次のようになります。

```text
Channel × Height × Width
1 × 28 × 28
```

PyTorchでは画像のChannelが最初に来ることに注意してください。


In [ ]:
# 画像をTensorに変換する処理
transform = transforms.ToTensor()

# Trainデータ
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

# Testデータ
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

print("Trainデータ数:", len(train_dataset))
print("Testデータ数 :", len(test_dataset))


## 5. 画像1枚とラベルを確認する

PyTorchのDatasetでは、

```python
image, label = dataset[index]
```

のように、画像と正解ラベルを1組として取り出せます。

ここでは先頭のデータを確認します。


In [ ]:
image, label = train_dataset[0]

print("画像の型:", type(image))
print("画像のshape:", image.shape)
print("pixel値の最小:", image.min().item())
print("pixel値の最大:", image.max().item())
print("正解ラベル:", label)

plt.figure(figsize=(3, 3))
plt.imshow(image.squeeze(0), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


### `squeeze(0)` は何をしている？

画像Tensorのshapeは、

```text
1 × 28 × 28
```

です。

先頭の `1` はGrayscale画像のChannel数です。

`matplotlib` で表示しやすくするために、

```python
image.squeeze(0)
```

でChannelの次元を一時的に取り除き、

```text
28 × 28
```

にしています。

モデルに入力するときは、Channel次元が必要です。


## 6. 複数のMNIST画像を見る

MNISTには同じ数字でも、さまざまな書き方があります。

CNNは多数の画像から、

- 線の形
- 曲がり方
- 数字を構成する局所的なパターン

を学習します。


In [ ]:
plt.figure(figsize=(10, 4))

for i in range(12):
    image, label = train_dataset[i]

    plt.subplot(2, 6, i + 1)
    plt.imshow(image.squeeze(0), cmap="gray")
    plt.title(f"Label: {label}")
    plt.axis("off")

plt.tight_layout()
plt.show()


# PART 2 — DatasetとDataLoader

## 7. DataLoaderとは

Datasetは画像を1枚ずつ管理します。

しかし、ニューラルネットワークの学習では、通常は複数の画像をまとめて処理します。

この画像のまとまりを **Batch** と呼びます。

```text
Dataset
  ↓ DataLoader
Batch 1：128枚
Batch 2：128枚
Batch 3：128枚
...
```

`DataLoader` は、次の処理を行います。

- Datasetからデータを取り出す
- Batchにまとめる
- Trainデータの順番をShuffleする


In [ ]:
BATCH_SIZE = 128

sample_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(sample_loader))

print("画像Batchのshape:", images.shape)
print("ラベルBatchのshape:", labels.shape)


画像Batchのshapeは次の順番です。

```text
Batch × Channel × Height × Width
```

MNISTでBatch Sizeが128なら、

```text
128 × 1 × 28 × 28
```

となります。

ラベルは画像1枚につき1個なので、

```text
128
```

です。


# PART 3 — CNNモデルを作る

## 8. 今回のCNN構造

今回作るCNNは、次の順番で画像を処理します。

```text
Input
1 × 28 × 28
      ↓
Conv2d：16 channels
ReLU
MaxPool
      ↓
16 × 14 × 14
      ↓
Conv2d：32 channels
ReLU
MaxPool
      ↓
32 × 7 × 7
      ↓
Flatten
      ↓
Linear：64
      ↓
Linear：10
      ↓
0〜9の10クラス
```

前半のConvolution部分が **Feature Extractor**、  
後半のLinear部分が **Classifier** です。


## 9. PyTorchでCNNを定義する

PyTorchでは、`nn.Module` を継承してモデルを作ります。

### 主なLayer

- `nn.Conv2d`：画像から局所的な特徴を抽出
- `nn.ReLU`：非線形変換
- `nn.MaxPool2d`：Feature Mapのサイズを小さくする
- `nn.Flatten`：3次元の特徴を1次元に変換
- `nn.Linear`：抽出された特徴からクラスを予測

### 最後にSoftmaxを書かない理由

このあと使う `CrossEntropyLoss` は、  
Softmax前の値である **logits** を直接受け取ります。

そのため、モデルの最後には `Softmax` を書きません。


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # 入力：1 × 28 × 28
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # 入力：16 × 14 × 14
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
            # 出力：32 × 7 × 7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


## 10. モデルを確認する

CNNにダミー画像を1枚入力して、

- 入力shape
- 出力shape

を確認します。

出力の10個の値は、0〜9の各クラスに対する **logits** です。


In [ ]:
model = SimpleCNN().to(device)

print(model)

dummy_image = torch.randn(1, 1, 28, 28).to(device)

with torch.no_grad():
    dummy_output = model(dummy_image)

print("\n入力shape:", dummy_image.shape)
print("出力shape:", dummy_output.shape)


出力shapeが

```text
1 × 10
```

であることを確認してください。

- `1`：入力した画像の枚数
- `10`：0〜9の10クラス

モデルは10個のlogitのうち、最も大きい値の位置を予測クラスとして使います。

```python
predicted_class = logits.argmax(dim=1)
```


# PART 4 — CNNの学習処理

## 11. LossとOptimizer

### Loss

今回は多クラス分類なので、

```python
nn.CrossEntropyLoss()
```

を使います。

PyTorchでは正解ラベルをone-hot encodingせず、  
`0, 1, 2, ..., 9` の整数のまま使用します。

### Optimizer

今回はAdamを使います。

```python
torch.optim.Adam
```

Optimizerは、Lossが小さくなるようにモデルのWeightを更新します。


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss     :", criterion)
print("Optimizer:", optimizer.__class__.__name__)


## 12. 1回の学習で行う処理

1つのBatchに対して、次の順番で学習します。

```text
① Gradientをリセット
② Forwardして予測
③ Lossを計算
④ BackwardでGradientを計算
⑤ OptimizerでWeightを更新
```

対応するPyTorchコードは次の4行が中心です。

```python
optimizer.zero_grad()
logits = model(images)
loss.backward()
optimizer.step()
```


## 13. 1 Epochを学習する関数

同じ学習処理を5-Foldの各Foldで使うため、関数にまとめます。

この関数は、1 Epochの

- 平均Loss
- Accuracy

を返します。


In [ ]:
def train_one_epoch(model, data_loader, optimizer, criterion, device):
    # 学習モード
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        # ① 前のBatchのGradientをリセット
        optimizer.zero_grad()

        # ② Forward
        logits = model(images)

        # ③ Lossを計算
        loss = criterion(logits, labels)

        # ④ Backward
        loss.backward()

        # ⑤ Weightを更新
        optimizer.step()

        # 統計を記録
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy


## 14. Validation / Testを評価する関数

評価するときはWeightを更新しません。

そのため、

```python
model.eval()
torch.no_grad()
```

を使います。

- `model.eval()`：評価モードにする
- `torch.no_grad()`：Gradientを計算しない

この関数も平均LossとAccuracyを返します。


In [ ]:
def evaluate(model, data_loader, criterion, device):
    # 評価モード
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size

            predictions = logits.argmax(dim=1)

            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += batch_size

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy


# PART 5 — 5-Fold Cross Validation

## 15. なぜCross Validationを行うのか

Trainデータを一度だけTrain / Validationに分けると、  
偶然どのデータがValidationに入ったかによって結果が変わる可能性があります。

5-Fold Cross Validationでは、Trainデータを5つに分けます。

```text
Fold 1：Validation | Train | Train | Train | Train
Fold 2：Train | Validation | Train | Train | Train
Fold 3：Train | Train | Validation | Train | Train
Fold 4：Train | Train | Train | Validation | Train
Fold 5：Train | Train | Train | Train | Validation
```

各データは、

- 4回：学習用
- 1回：Validation用

として使われます。


## 16. Testデータは5-Foldに入れない

今回のデータの使い方は次のとおりです。

```text
MNIST Train：60,000枚
    │
    └── 5-Fold Cross Validation
          ├── Fold 1
          ├── Fold 2
          ├── Fold 3
          ├── Fold 4
          └── Fold 5

MNIST Test：10,000枚
    │
    └── 最後に1回だけ評価
```

Testデータをモデル選択やFoldの評価に繰り返し使うと、  
Testデータに合わせた判断をしてしまいます。

そのため、Testデータは最後まで使用しません。


## 17. ClassificationではStratifiedKFoldを使う

数字0〜9の割合がFoldごとに大きく変わると、  
Fold間の比較が不公平になる可能性があります。

そこで今回は、

```python
StratifiedKFold
```

を使います。

StratifiedKFoldは、各数字クラスの割合をできるだけ保ちながら  
データを5分割します。


## 18. 授業用の実行設定

CNNを5回学習するため、全60,000枚を使うと時間がかかります。

そこで、初期設定では次の **FAST MODE** を使います。

```text
5-Foldに使う画像：10,000枚
FoldごとのEpoch：1
最終モデルのEpoch：2
```

時間に余裕がある場合は、

```python
FAST_MODE = False
```

に変更できます。

> FAST MODEでも、5-Foldの分割方法と学習手順は同じです。


In [ ]:
FAST_MODE = True

BATCH_SIZE = 128
LEARNING_RATE = 0.001
N_SPLITS = 5

if FAST_MODE:
    CV_SAMPLE_SIZE = 10_000
    EPOCHS_PER_FOLD = 1
    FINAL_EPOCHS = 2
else:
    CV_SAMPLE_SIZE = len(train_dataset)
    EPOCHS_PER_FOLD = 2
    FINAL_EPOCHS = 5

print("FAST_MODE       :", FAST_MODE)
print("CV_SAMPLE_SIZE  :", CV_SAMPLE_SIZE)
print("EPOCHS_PER_FOLD :", EPOCHS_PER_FOLD)
print("FINAL_EPOCHS    :", FINAL_EPOCHS)


## 19. 5-Foldに使うデータを準備する

FAST MODEでは、60,000枚から10,000枚を選びます。

このとき `stratify` を使い、  
数字0〜9の割合を保つように選びます。

`train_dataset.targets` には、Train画像の正解ラベルが入っています。


In [ ]:
all_targets = np.array(train_dataset.targets)
all_indices = np.arange(len(train_dataset))

if CV_SAMPLE_SIZE < len(train_dataset):
    cv_indices, _ = train_test_split(
        all_indices,
        train_size=CV_SAMPLE_SIZE,
        stratify=all_targets,
        random_state=SEED
    )
else:
    cv_indices = all_indices

cv_targets = all_targets[cv_indices]

print("5-Foldに使う画像数:", len(cv_indices))

class_counts = pd.Series(
    cv_targets
).value_counts().sort_index()

display(
    pd.DataFrame({
        "数字": class_counts.index,
        "画像数": class_counts.values
    })
)


## 20. 5-Fold Cross Validationを実行する

各Foldでは、必ず次のものを新しく作り直します。

- CNNモデル
- Optimizer
- Train DataLoader
- Validation DataLoader

### なぜモデルを作り直すのか

前のFoldで学習したWeightを次のFoldに引き継ぐと、  
次のValidationデータをすでに学習した状態になる可能性があります。

そのためFoldごとに、

```python
model = SimpleCNN()
```

から再スタートします。


In [ ]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_results = []

for fold, (train_position, val_position) in enumerate(
    skf.split(
        np.zeros(len(cv_targets)),
        cv_targets
    ),
    start=1
):
    print("=" * 60)
    print(f"Fold {fold} / {N_SPLITS}")

    # cv_indices内の位置を、元Datasetのindexに戻す
    fold_train_indices = cv_indices[train_position]
    fold_val_indices = cv_indices[val_position]

    fold_train_dataset = Subset(
        train_dataset,
        fold_train_indices
    )

    fold_val_dataset = Subset(
        train_dataset,
        fold_val_indices
    )

    train_loader = DataLoader(
        fold_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    val_loader = DataLoader(
        fold_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    # FoldごとにSeed、モデル、Optimizerを作り直す
    set_seed(SEED + fold)

    fold_model = SimpleCNN().to(device)

    fold_optimizer = torch.optim.Adam(
        fold_model.parameters(),
        lr=LEARNING_RATE
    )

    fold_criterion = nn.CrossEntropyLoss()

    for epoch in range(1, EPOCHS_PER_FOLD + 1):
        train_loss, train_accuracy = train_one_epoch(
            fold_model,
            train_loader,
            fold_optimizer,
            fold_criterion,
            device
        )

        val_loss, val_accuracy = evaluate(
            fold_model,
            val_loader,
            fold_criterion,
            device
        )

        print(
            f"Epoch {epoch}/{EPOCHS_PER_FOLD} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy * 100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy * 100:.2f}%"
        )

    fold_results.append({
        "Fold": fold,
        "Train Accuracy": train_accuracy,
        "Validation Accuracy": val_accuracy,
        "Validation Loss": val_loss
    })


## 21. 5つのFold結果をまとめる

1回だけのValidation Accuracyではなく、  
5回の結果から次を計算します。

- 平均 Accuracy
- 標準偏差

平均は全体的な性能を表します。

標準偏差が小さい場合、Foldが変わっても比較的安定した結果だったと考えられます。


In [ ]:
fold_results_df = pd.DataFrame(fold_results)

fold_results_df["Train Accuracy (%)"] = (
    fold_results_df["Train Accuracy"] * 100
)

fold_results_df["Validation Accuracy (%)"] = (
    fold_results_df["Validation Accuracy"] * 100
)

display(
    fold_results_df[
        [
            "Fold",
            "Train Accuracy (%)",
            "Validation Accuracy (%)",
            "Validation Loss"
        ]
    ].round(3)
)

mean_val_accuracy = (
    fold_results_df["Validation Accuracy"].mean()
)

std_val_accuracy = (
    fold_results_df["Validation Accuracy"].std(ddof=0)
)

print(
    "5-Fold Validation Accuracy:",
    f"{mean_val_accuracy * 100:.2f}%"
    f" ± {std_val_accuracy * 100:.2f}%"
)


## 22. FoldごとのAccuracyをグラフで見る

Fold間でAccuracyが大きく変化していないか確認します。

横軸：Fold番号  
縦軸：Validation Accuracy


In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    fold_results_df["Fold"],
    fold_results_df["Validation Accuracy (%)"]
)

plt.axhline(
    fold_results_df["Validation Accuracy (%)"].mean(),
    linestyle="--",
    label="Mean Accuracy"
)

plt.xlabel("Fold")
plt.ylabel("Validation Accuracy (%)")
plt.title("5-Fold Cross Validation")
plt.xticks(range(1, N_SPLITS + 1))
plt.legend()
plt.show()


# PART 6 — 最終モデルを作る

## 23. Cross Validationの後に何をする？

5-Fold Cross Validationによって、  
今回のCNN構造と学習設定のValidation性能を確認しました。

次に、新しいモデルを作り、

```text
MNIST Train 60,000枚すべて
```

を使って学習します。

そのあと、今まで使っていなかった

```text
MNIST Test 10,000枚
```

で最後に1回だけ評価します。


In [ ]:
final_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

set_seed(SEED)

final_model = SimpleCNN().to(device)

final_criterion = nn.CrossEntropyLoss()

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LEARNING_RATE
)

print("最終モデルを新しく作成しました。")


## 24. 全Trainデータで最終モデルを学習する

ここではTestデータをまだ使いません。

各Epochについて、

- Train Loss
- Train Accuracy

を表示します。


In [ ]:
final_history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    train_loss, train_accuracy = train_one_epoch(
        final_model,
        final_train_loader,
        final_optimizer,
        final_criterion,
        device
    )

    final_history.append({
        "Epoch": epoch,
        "Train Loss": train_loss,
        "Train Accuracy": train_accuracy
    })

    print(
        f"Epoch {epoch}/{FINAL_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy * 100:.2f}%"
    )


## 25. Testデータで最終評価する

Testデータは、モデル選択にも5-Foldにも使っていません。

ここで初めてTestデータを使い、  
未知の画像に対する最終性能を確認します。


In [ ]:
test_loss, test_accuracy = evaluate(
    final_model,
    test_loader,
    final_criterion,
    device
)

print("Test Loss    :", round(test_loss, 4))
print(
    "Test Accuracy:",
    f"{test_accuracy * 100:.2f}%"
)


# PART 7 — 予測結果を詳しく見る

## 26. Test画像の予測結果を集める

Confusion Matrixや間違った画像を確認するため、  
すべてのTest画像について、

- 正解ラベル
- 予測ラベル
- 画像Tensor

を集めます。


In [ ]:
def collect_predictions(model, data_loader, device):
    model.eval()

    all_images = []
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in data_loader:
            logits = model(
                images.to(device)
            )

            predictions = logits.argmax(
                dim=1
            ).cpu()

            all_images.append(images.cpu())
            all_labels.append(labels.cpu())
            all_predictions.append(predictions)

    all_images = torch.cat(all_images)
    all_labels = torch.cat(all_labels)
    all_predictions = torch.cat(all_predictions)

    return (
        all_images,
        all_labels,
        all_predictions
    )

test_images, test_labels, test_predictions = (
    collect_predictions(
        final_model,
        test_loader,
        device
    )
)

print("画像:", test_images.shape)
print("正解:", test_labels.shape)
print("予測:", test_predictions.shape)


## 27. Confusion Matrix

Confusion Matrixでは、

- 行：実際の数字
- 列：予測した数字

を表します。

対角線上の値は正解です。

対角線以外を見ると、  
どの数字をどの数字と間違えたか確認できます。


In [ ]:
cm = confusion_matrix(
    test_labels.numpy(),
    test_predictions.numpy(),
    labels=list(range(10))
)

fig, ax = plt.subplots(figsize=(8, 8))

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(10))
)

display_cm.plot(
    ax=ax,
    colorbar=False
)

plt.title("MNIST Confusion Matrix")
plt.show()


## 28. 数字ごとのAccuracyを確認する

全体Accuracyだけでなく、  
各数字をどの程度正しく分類できたかを計算します。

例えば、

- 5だけ精度が低い
- 4と9を間違えやすい

などの特徴が見つかる可能性があります。


In [ ]:
per_class_results = []

for digit in range(10):
    digit_mask = (
        test_labels == digit
    )

    digit_total = digit_mask.sum().item()

    digit_correct = (
        test_predictions[digit_mask] == digit
    ).sum().item()

    digit_accuracy = (
        digit_correct / digit_total
    )

    per_class_results.append({
        "Digit": digit,
        "Test images": digit_total,
        "Correct": digit_correct,
        "Accuracy (%)": digit_accuracy * 100
    })

per_class_df = pd.DataFrame(
    per_class_results
)

display(
    per_class_df.round(2)
)


## 29. 間違えた画像を見る

モデルが間違えたTest画像を取り出します。

画像を見ながら、

- 人間にも読みにくいか
- 別の数字に見える部分があるか
- どの数字同士を間違えやすいか

を考えてみましょう。


In [ ]:
wrong_positions = torch.where(
    test_labels != test_predictions
)[0]

print(
    "間違えた画像数:",
    len(wrong_positions)
)

show_count = min(
    10,
    len(wrong_positions)
)

set_seed(SEED)

selected_positions = wrong_positions[
    torch.randperm(
        len(wrong_positions)
    )[:show_count]
]

plt.figure(figsize=(12, 5))

for i, position in enumerate(selected_positions):
    image = test_images[position]
    actual = test_labels[position].item()
    predicted = test_predictions[position].item()

    plt.subplot(2, 5, i + 1)
    plt.imshow(
        image.squeeze(0),
        cmap="gray"
    )
    plt.title(
        f"Actual: {actual}\n"
        f"Pred: {predicted}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()


## 30. 画像1枚を予測し、確率を見る

モデルの出力はlogitsです。

人が読みやすい確率に変換するときだけ、  
推論後に `softmax()` を使います。

```text
logits
  ↓ softmax
0〜9の予測確率
```

ここではTest画像1枚の予測確率を確認します。


In [ ]:
sample_image, sample_label = test_dataset[0]

final_model.eval()

with torch.no_grad():
    sample_logits = final_model(
        sample_image.unsqueeze(0).to(device)
    )

    sample_probabilities = torch.softmax(
        sample_logits,
        dim=1
    ).cpu().squeeze(0)

sample_prediction = (
    sample_probabilities.argmax().item()
)

plt.figure(figsize=(10, 3))

plt.subplot(1, 2, 1)
plt.imshow(
    sample_image.squeeze(0),
    cmap="gray"
)
plt.title(
    f"Actual: {sample_label}\n"
    f"Predicted: {sample_prediction}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(
    range(10),
    sample_probabilities.numpy()
)
plt.xlabel("Digit")
plt.ylabel("Probability")
plt.title("Prediction Probability")
plt.xticks(range(10))

plt.tight_layout()
plt.show()


# まとめ

今回の実習では、PyTorchを使ってMNISTの0〜9を分類しました。

## CNNの基本的な流れ

```text
MNIST画像
   ↓
Dataset
   ↓
DataLoader
   ↓
CNN
   ↓
CrossEntropyLoss
   ↓
Backward
   ↓
Optimizer
   ↓
数字0〜9を予測
```

## 5-Fold Cross Validation

```text
Trainデータを5分割
   ↓
4つで学習・1つでValidation
   ↓
Validation部分を交代して5回学習
   ↓
平均Accuracyと標準偏差
```

## 最後のTest

```text
5-FoldではTestを使わない
   ↓
最終モデルをTrain全体で学習
   ↓
Testデータで1回だけ最終評価
```


# 確認問題

実行結果を見ながら、次の問いに答えてください。

### Q1
今回の入力 **X** と正解 **y** は何ですか。

### Q2
MNIST画像1枚のTensor shapeは何ですか。  
また、各数字は何を意味しますか。

### Q3
Batch Sizeが128の場合、画像Batchのshapeは何ですか。

### Q4
今回のCNNには、Convolution Layerが何層ありますか。

### Q5
モデルの最後にSoftmaxを書かなかった理由を説明してください。

### Q6
5つのFoldそれぞれのValidation Accuracyを書いてください。

### Q7
5-Fold Validation Accuracyの平均と標準偏差を書いてください。

### Q8
最終的なTest Accuracyは何％でしたか。

### Q9
Confusion Matrixから、間違えやすい数字の組み合わせを1つ挙げてください。

### Q10
間違えた画像を見て、モデルが間違えた理由として考えられることを1つ書いてください。


# 発展課題（時間に余裕がある場合）

次のうち1つを変更し、結果を比較してください。

## A. Epoch数を変更する

```python
EPOCHS_PER_FOLD = 2
FINAL_EPOCHS = 5
```

## B. Filter数を増やす

```python
16 → 32
32 → 64
```

## C. Learning Rateを変更する

```python
0.001 → 0.0005
```

## D. Dropoutを追加する

```python
nn.Dropout(0.3)
```

### 比較する項目

- 5-Fold平均Accuracy
- 標準偏差
- Test Accuracy
- 学習時間
- 間違えた画像


## 参考

- PyTorch
- torchvision MNIST Dataset
- PyTorch `nn.Conv2d`
- PyTorch `nn.CrossEntropyLoss`
- scikit-learn `StratifiedKFold`

このノートブックでは、CNNと5-Fold Cross Validationの基本を理解するため、  
モデル構造と学習設定をできるだけ単純にしています。
